# Phase 2 — Heuristique VRPTW : NNH + 2-opt
### Nearest Neighbor Heuristic + Amélioration Locale

---

> **Objectif de ce notebook**  
> Comprendre, de zéro, pourquoi et comment on résout le VRPTW avec une heuristique.  
> Ce document explique la logique, les mathématiques, les analogies et le code — avant de tout assembler.

---

## Table des matières

1. [C'est quoi une heuristique ?](#1)
2. [Rappel du problème VRPTW](#2)
3. [Étape 1 — Nearest Neighbor Heuristic (NNH)](#3)
4. [Étape 2 — Amélioration locale 2-opt](#4)
5. [Les contraintes VRPTW dans le code](#5)
6. [Implémentation complète commentée](#6)
7. [Validation et résultats](#7)

<a id='1'></a>
## 1. C'est quoi une heuristique ?

### Le problème de fond

Pour $n$ clients, le nombre de tournées possibles est $(n-1)!$

| n clients | Tournées possibles | Temps brute-force (à 1 milliard op/s) |
|-----------|-------------------|----------------------------------------|
| 10        | 362 880           | < 1 ms |
| 20        | $1.2 \times 10^{17}$ | ~4 ans |
| 50        | $3 \times 10^{62}$ | âge de l'univers × $10^{44}$ |
| 100       | $10^{157}$        | incalculable |

Tester toutes les possibilités est **impossible**. Il faut trouver une bonne solution sans tout tester.

### Définition

> Une **heuristique** est une méthode qui produit une solution **bonne et rapide**, sans garantie d'optimalité, en utilisant des **règles de bon sens**.

### L'analogie de la chambre en désordre

**Méthode exacte :** mesurer chaque objet, calculer la disposition optimale au millimètre — ça prend 3 heures.  
**Heuristique :** ranger d'abord ce qui traîne au milieu, puis ce qui est visible, puis le reste — ça prend 5 minutes, c'est raisonnable.

Notre heuristique fonctionne en **deux temps** :
- **NNH** : construire rapidement une solution valide ("ranger en mode brouillon")
- **2-opt** : améliorer cette solution sans tout refaire ("réajuster ce qui n'est pas bien rangé")

### Position dans le projet

```
NNH + 2-opt  →  Baseline rapide      (~5-10% sous-optimal)
Recuit simulé →  Meilleure qualité   (~1-5% sous-optimal)   [Rayene]
Deep Learning →  Inférence constante (~variable)            [Victor]
```

Notre heuristique est la **référence de base** : si le recuit ou le DL font moins bien, il y a un problème.

<a id='2'></a>
## 2. Rappel du problème VRPTW

### Le cadre mathématique

On a un graphe orienté $G = (V, A, c)$ où :
- $V = \{0, 1, ..., n\}$ : ensemble des nœuds ($0$ = dépôt, $1..n$ = clients)
- $A$ : ensemble des arcs (on peut aller de n'importe quel nœud à n'importe quel autre)
- $c_{ij}$ : coût (distance euclidienne) de $i$ vers $j$

### Les variables

$$x_{ijk} = \begin{cases} 1 & \text{si le véhicule } k \text{ emprunte l'arc } i \to j \\ 0 & \text{sinon} \end{cases}$$

$$t_{ik} = \text{heure d'arrivée du véhicule } k \text{ au client } i$$

### L'objectif

$$\min \sum_{k} \sum_{i} \sum_{j} c_{ij} \cdot x_{ijk}$$

Minimiser la **distance totale** parcourue par l'ensemble des véhicules.

### Les 5 contraintes

| # | Contrainte | Formulation | En clair |
|---|------------|-------------|----------|
| C1 | Couverture | $\sum_{k} \sum_{j} x_{ijk} = 1 \quad \forall i \in V \setminus \{0\}$ | Chaque client est livré **exactement une fois** |
| C2 | Capacité | $\sum_{i} d_i \cdot \sum_{j} x_{ijk} \leq Q \quad \forall k$ | Un véhicule ne peut pas dépasser sa **capacité** $Q$ |
| C3 | Cohérence | Si $k$ arrive en $i$, il repart de $i$ | Un véhicule qui livre $i$ **repart de $i$** |
| C4 | Fenêtre temporelle | $a_i \leq t_{ik} \leq b_i$ | On arrive chez le client dans son **créneau** $[a_i, b_i]$ |
| C5 | Horizon | $t_{ik} + s_i + c_{i,depot} \leq H$ | Le véhicule rentre au dépôt avant la fin de journée $H$ |

Avec $d_i$ la demande du client $i$, $s_i$ son temps de service, $H = 480$ min (8h).

### Ce que notre heuristique doit respecter

À chaque décision, on vérifie **C1, C2, C4 et C5**. C3 est garantie structurellement par la façon dont on construit les routes.

<a id='3'></a>
## 3. Étape 1 — Nearest Neighbor Heuristic (NNH)

### L'analogie du livreur naïf

Imagine un livreur qui part du dépôt. Il est **pressé et un peu naïf** : à chaque arrêt, il regarde autour de lui et se demande :

> *"Quel est le prochain client non encore livré le plus proche de moi, que je peux encore atteindre ?"*

Il y va. Puis il recommence. Jusqu'à ce que :
- Son camion soit plein (contrainte capacité), **ou**
- Il ne puisse plus atteindre aucun client dans les temps (contrainte TW), **ou**
- Il n'y ait plus de clients à livrer.

Alors il rentre au dépôt, et un **nouveau véhicule** repart pour les clients restants.

### La complexité : $O(n^2)$

Pour chaque client à placer ($n$ clients), on examine tous les clients restants ($n$ au pire) pour trouver le plus proche.  
→ $n \times n = O(n^2)$ opérations. Comparé à $O(n!)$ du brute-force : **extrêmement rapide**.

### L'algorithme pas à pas

```
INITIALISATION :
    position_courante  ← dépôt (nœud 0)
    heure_courante     ← 0
    capacite_restante  ← Q
    clients_non_livres ← {1, 2, ..., n}
    route_courante     ← []

TANT QUE clients_non_livres n'est pas vide :
    meilleur_client  ← None
    meilleure_dist   ← +∞

    POUR chaque client i dans clients_non_livres :
        SI faisable(i, position_courante, heure_courante, capacite_restante) :
            SI distance(position_courante, i) < meilleure_dist :
                meilleur_client ← i
                meilleure_dist  ← distance(position_courante, i)

    SI meilleur_client trouvé :
        Ajouter meilleur_client à route_courante
        Mettre à jour heure, capacité, position
        Retirer meilleur_client de clients_non_livres
    SINON :
        Fermer route_courante (retour dépôt)
        Ouvrir nouvelle route (nouveau véhicule)

RETOURNER toutes les routes
```

### La fonction de faisabilité — le cœur du respect des contraintes

Avant de choisir un client $j$ depuis la position $i$ à l'heure $t$ :

**Étape 1 — Contrainte C2 (capacité) :**
$$d_j \leq Q_{restante}$$

**Étape 2 — Calculer l'heure d'arrivée :**
$$t_{arrivee} = \max(t + c_{ij},\; a_j)$$
Le $\max$ gère l'attente : si on arrive **avant** l'ouverture $a_j$, on attend.

**Étape 3 — Contrainte C4 (fenêtre temporelle) :**
$$t_{arrivee} \leq b_j$$
On ne peut pas arriver **après** la fermeture.

**Étape 4 — Contrainte C5 (horizon) :**
$$t_{arrivee} + s_j + c_{j,0} \leq H$$
Après avoir livré $j$, on doit pouvoir **rentrer au dépôt** avant la fin de journée.

### Le problème du NNH : l'horizon local

Le livreur naïf peut se retrouver **bloqué** : en choisissant toujours le plus proche, il peut aller vers des clients isolés qui l'éloignent de groupes entiers.  
C'est pour ça qu'on ajoute le 2-opt.

<a id='4'></a>
## 4. Étape 2 — Amélioration locale 2-opt

### L'analogie du GPS qui recalcule

Tu conduis et soudain le GPS dit : *"J'ai trouvé un meilleur chemin — si tu inverses ce tronçon, tu gagnes 8 minutes."*

Le 2-opt fait exactement ça : il examine la route construite par NNH et cherche des **portions à inverser** qui raccourciraient le trajet.

### Pourquoi des routes se croisent ?

NNH peut produire des routes qui se **croisent visuellement** sur une carte.  
C'est toujours sous-optimal : deux segments qui se croisent peuvent **toujours** être remplacés par deux segments qui ne se croisent pas, pour un coût total inférieur.

```
AVANT (route avec croisement) :          APRÈS (2-opt appliqué) :

Dépôt                                    Dépôt
  |                                        |
  A -----> B                               A -----> C
           |      <- croisement ->                  |
  D <----- C                               D <----- B
  |                                        |
 Dépôt                                   Dépôt

Coût : A→B + C→D                         Coût : A→C + B→D  (plus court !)
```

**Propriété clé :** Inverser le segment entre $i$ et $j$ dans une route revient à **supprimer deux arcs** et les **remplacer par deux autres**.

### Les mathématiques du 2-opt

Soit une route $R = [r_0, r_1, ..., r_{n-1}]$.  
On considère deux indices $i < j$.  
Le **gain** obtenu en inversant le segment $[i, j]$ est :

$$\Delta = c_{r_{i-1}, r_i} + c_{r_j, r_{j+1}} - c_{r_{i-1}, r_j} - c_{r_i, r_{j+1}}$$

Si $\Delta > 0$ → l'inversion **raccourcit** la route → on l'accepte.

La nouvelle route est :
$$R' = [r_0, ..., r_{i-1},\; r_j, r_{j-1}, ..., r_i,\; r_{j+1}, ..., r_{n-1}]$$

(le segment entre $i$ et $j$ est **renversé**)

### L'algorithme 2-opt

```
RÉPÉTER :
    amelioration_trouvee ← Faux

    POUR chaque route R dans la solution :
        POUR chaque paire (i, j) avec i < j dans R :
            Calculer Δ = gain de l'inversion du segment [i, j]
            SI Δ > 0 ET nouvelle route valide (TW + capacité) :
                Inverser R[i:j+1]
                amelioration_trouvee ← Vrai
                SORTIR des boucles internes (recommencer)

JUSQU'À ce qu'aucune amélioration ne soit trouvée
→ on est dans un optimum local 2-opt
```

### Complexité du 2-opt

Pour une route de $m$ clients : $\binom{m}{2} = O(m^2)$ paires à tester par itération.  
Nombre d'itérations : au plus $O(m^2)$ (chaque amélioration est strictement meilleure).  
**Total par route :** $O(m^4)$ dans le pire cas — mais en pratique **beaucoup plus rapide**.

### La contrainte des fenêtres temporelles dans le 2-opt

Avec le VRPTW, une inversion qui **raccourcit** la route peut tout de même la **rendre invalide** si elle change l'ordre des visites et fait rater des créneaux horaires.  
→ Après chaque inversion candidate, on **revalide** la route complète avant de l'accepter.

<a id='5'></a>
## 5. Les contraintes VRPTW dans le code

### La fonction `est_valide(route, instance)`

C'est la fonction la plus critique — elle est appelée **à chaque décision** dans NNH et à chaque **inversion candidate** dans le 2-opt.

Elle simule le trajet complet du véhicule sur la route et vérifie :

```
VÉRIFICATION 1 — Capacité (C3) :
    charge = somme des demandes des clients de la route
    SI charge > Q : INVALIDE

VÉRIFICATION 2 — Fenêtres temporelles + horizon (C4 + C5) :
    t ← 0  (on part du dépôt à t=0)
    position ← dépôt

    POUR chaque client c dans la route :
        t ← t + duree(position, c)       # on se déplace
        t ← max(t, a_c)                  # on attend si trop tôt
        SI t > b_c : INVALIDE            # on est trop tard → C4 violée
        t ← t + s_c                      # temps de service chez c
        position ← c

    SI t + duree(position, dépôt) > H : INVALIDE  # retour impossible → C5 violée

RETOURNER Vrai
```

### Le traitement du "wait" (attente)

Le $\max(t, a_j)$ est essentiel.  
Si on arrive à 8h30 et que le client ouvre à 9h, on **attend** 30 minutes.  
Ce temps d'attente est propagé : tous les clients suivants dans la route seront visités plus tard.  
C'est pourquoi l'ordre des visites a un impact complexe sur la validité de la route.

### Tableau récapitulatif des 6 contraintes VRPTW

| Contrainte | Vérifiée dans NNH | Vérifiée dans 2-opt |
|------------|-------------------|---------------------|
| C1 — Couverture | ✅ structure (chaque client ajouté une seule fois) | ✅ l'inversion ne supprime aucun client |
| C2 — Flux conservation | ✅ structure (routes 0→…→0, implicite) | ✅ structure de l'algo |
| C3 — Capacité | ✅ vérifiée avant chaque ajout | ✅ `est_valide()` |
| C4 — Fenêtre TW | ✅ vérifiée avant chaque ajout | ✅ `est_valide()` |
| C5 — Horizon | ✅ vérifié avant chaque ajout | ✅ `est_valide()` |
| C6 — Anti-sous-cycles | ✅ structure (ensemble `clients_non_livres`) | ✅ structure de l'algo |

### Contrainte souple vs contrainte dure — choix de conception

Le notebook de modélisation (cellule 14) formule C4 comme une **contrainte souple** avec une pénalité $\lambda \cdot \max(0,\, t_i - b_i)$. Ce choix est pertinent pour le **recuit simulé** (Phase 3) : accepter temporairement des solutions légèrement infaisables permet d'explorer l'espace de recherche et d'échapper aux optima locaux.

Pour la **NNH**, ce raisonnement ne s'applique pas. La NNH construit la solution de façon **gloutonne et irréversible** : une violation de TW à l'étape $k$ ne peut pas être corrigée à l'étape $k+1$. Appliquer C4 comme contrainte dure (rejet immédiat du client) est donc la seule approche cohérente — et garantit que chaque solution produite est **directement livrable**.

> **Résumé du choix :**  
> - **NNH + 2-opt (Phase 2)** → C4 *dure* (rejet) : solution toujours valide, baseline claire.  
> - **Recuit simulé (Phase 3)** → C4 *souple* (pénalités) : exploration plus large, meilleure qualité.  
> Le FDR autorise explicitement les deux approches : *« pénalités **ou** rejet »*.

<a id='6'></a>
## 6. Implémentation complète commentée

In [ ]:
import time
import sys
import os

PHASE4 = os.path.normpath(os.path.join(os.path.abspath(''), '..', 'Phase 4'))
sys.path.insert(0, PHASE4)
from preprocess import generate_instance

sys.path.insert(0, os.path.abspath(''))
from heuristique import cout_solution, est_valide, construire_solution_nnh, ameliorer_2opt, resoudre_heuristique


### 6.1 — Calcul du coût d'une solution

In [ ]:
def cout_solution(routes, instance):
    """
    Calcule le cout total = somme des distances de chaque segment.
    Utilise la matrice dist pre-calculee par generate_instance().
    Unite spatiale (carte 100x100). Pour convertir en minutes : x 0.6.
    """
    dist  = instance['dist']
    total = 0.0
    for route in routes:
        if not route:
            continue
        total += dist[0, route[0]]
        for i in range(len(route) - 1):
            total += dist[route[i], route[i + 1]]
        total += dist[route[-1], 0]
    return total


### 6.2 — Vérification de validité d'une route

In [ ]:
def est_valide(route, instance):
    """
    Vérifie qu'une route respecte toutes les contraintes VRPTW.
    
    Simule le trajet complet du véhicule et contrôle :
    - C2 : capacité totale chargée ≤ Q
    - C4 : arrivée chez chaque client dans sa fenêtre [a_i, b_i]
    - C5 : retour au dépôt avant l'horizon H

    @param route    : list — liste ordonnée des clients du véhicule
    @param instance : dict
    @return         : bool — True si toutes les contraintes sont respectées
    """
    if not route:
        return True  # route vide = toujours valide

    durees   = instance['durees']       # matrice des temps de trajet
    tw       = instance['time_windows'] # tw[i] = [a_i, b_i]
    service  = instance['service_times']
    demands  = instance['demands']
    capacity = instance['capacity']
    horizon  = instance['horizon']      # 480 min = 8h

    # --- C2 : vérification capacité ---
    # On calcule la charge totale du véhicule pour cette route
    if sum(demands[c] for c in route) > capacity + 1e-6:
        return False

    # --- C4 + C5 : simulation du trajet ---
    t   = 0.0   # heure courante (on part à t=0)
    pos = 0     # position courante (0 = dépôt)

    for client in route:
        # Déplacement vers le client
        t = t + durees[pos, client]

        # Attente si on arrive avant l'ouverture (C4 début)
        # ex : arrivée à 8h30, ouverture à 9h → on attend jusqu'à 9h
        t = max(t, tw[client, 0])

        # Vérification fermeture (C4 fin)
        # ex : fermeture à 11h, on arrive à 11h30 → invalide
        if t > tw[client, 1] + 1e-6:
            return False

        # Temps de service chez le client
        t  += service[client]
        pos = client

    # C5 : retour au dépôt dans l'horizon
    if t + durees[pos, 0] > horizon + 1e-6:
        return False

    return True

### 6.3 — Nearest Neighbor Heuristic (NNH)

In [ ]:
def construire_solution_nnh(instance):
    """
    Construit une solution par NNH.
    Pre-detecte les clients infaisables (C3/C4/C5 impossibles meme depuis le depot)
    et les isole. Ouvre autant de vehicules que necessaire pour garantir C1.
    """
    n        = instance['n']
    dist     = instance['dist']
    durees   = instance['durees']
    tw       = instance['time_windows']
    service  = instance['service_times']
    demands  = instance['demands']
    capacity = instance['capacity']
    horizon  = instance['horizon']

    routes              = []
    clients_non_livres  = set(range(1, n + 1))
    clients_infaisables = []

    # Pre-detection : clients impossibles a servir meme avec un vehicule dedie depuis le depot
    # C3 : demande individuelle depasse la capacite totale du vehicule
    # C4 : fenetre temporelle fermee avant meme d'arriver depuis le depot a t=0
    # C5 : retour au depot impossible meme si on part immediatement
    for client in list(clients_non_livres):
        t_min  = max(durees[0, client], tw[client, 0])
        c3_ok  = demands[client] <= capacity + 1e-6
        tw_ok  = t_min <= tw[client, 1] + 1e-6
        c5_ok  = t_min + service[client] + durees[client, 0] <= horizon + 1e-6
        if not c3_ok or not tw_ok or not c5_ok:
            clients_infaisables.append(client)
            clients_non_livres.discard(client)

    while clients_non_livres:
        route             = []
        pos               = 0
        t                 = 0.0
        capacite_restante = capacity

        while True:
            meilleur_client = None
            meilleure_dist  = float('inf')
            for client in clients_non_livres:
                if demands[client] > capacite_restante + 1e-6:
                    continue
                t_arrivee = max(t + durees[pos, client], tw[client, 0])
                if t_arrivee > tw[client, 1] + 1e-6:
                    continue
                if t_arrivee + service[client] + durees[client, 0] > horizon + 1e-6:
                    continue
                d = dist[pos, client]
                if d < meilleure_dist:
                    meilleure_dist  = d
                    meilleur_client = client
            if meilleur_client is None:
                break
            route.append(meilleur_client)
            clients_non_livres.discard(meilleur_client)
            t_arrivee         = max(t + durees[pos, meilleur_client], tw[meilleur_client, 0])
            t                 = t_arrivee + service[meilleur_client]
            capacite_restante -= demands[meilleur_client]
            pos                = meilleur_client

        if route:
            routes.append(route)
        else:
            # Securite : clients restants inaccessibles -> infaisables pour garantir C1
            clients_infaisables.extend(clients_non_livres)
            break

    if clients_infaisables:
        routes.append(clients_infaisables)

    return routes

### 6.4 — Amélioration locale 2-opt

In [ ]:
def ameliorer_2opt(routes, instance):
    """
    Améliore une solution par l'heuristique 2-opt intra-route.
    
    Pour chaque route, on teste toutes les paires d'arcs (i, j).
    Si inverser le segment [i, j] raccourcit la route ET reste valide
    au regard des contraintes VRPTW, on effectue l'inversion.
    
    On répète jusqu'à ce qu'aucune amélioration ne soit possible
    (optimum local 2-opt).

    @param routes   : list of lists — solution produite par NNH
    @param instance : dict
    @return         : list of lists — solution améliorée
    """
    dist = instance['dist']

    amelioration_globale = True

    while amelioration_globale:
        amelioration_globale = False

        for idx_route, route in enumerate(routes):
            n = len(route)
            if n < 3:          # besoin d'au moins 3 clients pour un échange utile
                continue

            amelioration_route = True

            while amelioration_route:
                amelioration_route = False

                for i in range(n - 1):
                    for j in range(i + 2, n):

                        # Noeuds concernés par l'échange :
                        # avant l'inversion : ...→ route[i] → route[i+1] → ... → route[j] → route[j+1] → ...
                        # après l'inversion : ...→ route[i] → route[j] → ... → route[i+1] → route[j+1] → ...
                        
                        # Noeuds "avant" et "après" le segment inversé
                        # (avec gestion des bords : dépôt = nœud 0)
                        node_avant_i  = 0 if i == 0 else route[i - 1]
                        node_apres_j  = 0 if j == n - 1 else route[j + 1]

                        # Calcul du gain Δ
                        # Δ > 0 → l'inversion raccourcit la route
                        cout_avant   = dist[node_avant_i, route[i]] + dist[route[j], node_apres_j]
                        cout_apres   = dist[node_avant_i, route[j]] + dist[route[i], node_apres_j]
                        delta        = cout_avant - cout_apres

                        if delta > 1e-6:   # amélioration trouvée

                            # Construire la route candidate avec le segment inversé
                            route_candidate = route[:i] + route[i:j+1][::-1] + route[j+1:]

                            # Valider les contraintes VRPTW avant d'accepter
                            # (une inversion peut raccourcir la distance mais violer les TW)
                            if est_valide(route_candidate, instance):
                                routes[idx_route]  = route_candidate
                                route              = route_candidate
                                amelioration_route = True
                                amelioration_globale = True
                                break   # recommencer depuis le début pour cette route

                    if amelioration_route:
                        break

    return routes

### 6.5 — Fonction principale

In [ ]:
def resoudre_heuristique(instance):
    """
    Resout une instance VRPTW par NNH + 2-opt.
    Retourne (routes, cout_final, stats).
    stats inclut : temps, couts, gain_2opt, n_vehicules, n_clients_infaisables, valide.
    """
    t0     = time.time()
    routes = construire_solution_nnh(instance)
    t_nnh  = time.time() - t0

    cout_initial = cout_solution(routes, instance)

    t1     = time.time()
    routes = ameliorer_2opt(routes, instance)
    t_2opt = time.time() - t1

    cout_final       = cout_solution(routes, instance)
    gain_pct         = (cout_initial - cout_final) / cout_initial * 100 if cout_initial > 0 else 0.0
    n_vehicules      = sum(1 for r in routes if r)
    routes_invalides = [r for r in routes if r and not est_valide(r, instance)]
    n_infaisables    = sum(len(r) for r in routes_invalides)
    valide           = len(routes_invalides) == 0

    stats = {
        'temps_nnh'             : round(t_nnh, 4),
        'temps_2opt'            : round(t_2opt, 4),
        'temps_total'           : round(t_nnh + t_2opt, 4),
        'cout_initial'          : round(cout_initial, 2),
        'cout_final'            : round(cout_final, 2),
        'gain_2opt_pct'         : round(gain_pct, 2),
        'n_vehicules_utilises'  : n_vehicules,
        'n_clients_infaisables' : n_infaisables,
        'valide'                : valide,
    }
    return routes, cout_final, stats


<a id='7'></a>
## 7. Validation et résultats

### 7.1 — Test sur petite instance

In [ ]:
instance_10 = generate_instance(n=10, n_vehicles=3, seed=42)
routes, cout, stats = resoudre_heuristique(instance_10)

print('=' * 52)
print('Instance : 10 clients, 3 vehicules, seed=42')
print('=' * 52)
for k, route in enumerate(routes):
    if route:
        print(f"  Vehicule {k+1} : depot -> {' -> '.join(map(str, route))} -> depot")
print()
print(f"Cout initial (NNH seul)    : {stats['cout_initial']}")
print(f"Cout final   (+ 2-opt)     : {stats['cout_final']}")
print(f"Gain du 2-opt              : {stats['gain_2opt_pct']} %")
print(f"Temps NNH                  : {stats['temps_nnh']} s")
print(f"Temps 2-opt                : {stats['temps_2opt']} s")
print(f"Temps total                : {stats['temps_total']} s")
print(f"Vehicules utilises         : {stats['n_vehicules_utilises']}")
print(f"Clients infaisables        : {stats['n_clients_infaisables']}")
print(f"Solution valide            : {stats['valide']}")


### 7.2 — Montée en charge : n = 10, 20, 50, 100, 200

In [ ]:
import matplotlib.pyplot as plt

tailles    = [10, 20, 50, 100, 200]
couts      = []
temps_list = []
gains      = []

print(f"{'n':>5} | {'Coût':>10} | {'Temps (s)':>10} | {'Gain 2-opt':>10} | {'Valide':>7}")
print("-" * 55)

for n in tailles:
    inst = generate_instance(n=n, n_vehicles=max(3, n // 10), seed=42)
    _, cout_n, stats_n = resoudre_heuristique(inst)
    couts.append(cout_n)
    temps_list.append(stats_n['temps_total'])
    gains.append(stats_n['gain_2opt_pct'])
    print(f"{n:>5} | {cout_n:>10.2f} | {stats_n['temps_total']:>10.4f} | {stats_n['gain_2opt_pct']:>9.2f}% | {str(stats_n['valide']):>7}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Graphe 1 : Coût total en fonction de n
axes[0].plot(tailles, couts, 'o-', color='steelblue', linewidth=2, markersize=8)
axes[0].set_title('Coût total vs taille instance')
axes[0].set_xlabel('Nombre de clients (n)')
axes[0].set_ylabel('Distance totale')
axes[0].grid(True, alpha=0.3)

# Graphe 2 : Temps de calcul en fonction de n
axes[1].plot(tailles, temps_list, 's-', color='tomato', linewidth=2, markersize=8)
axes[1].set_title('Temps de calcul vs taille instance')
axes[1].set_xlabel('Nombre de clients (n)')
axes[1].set_ylabel('Temps (secondes)')
axes[1].grid(True, alpha=0.3)

# Graphe 3 : Gain du 2-opt en fonction de n
axes[2].bar(tailles, gains, color='mediumseagreen', alpha=0.8, width=8)
axes[2].set_title('Gain apporté par le 2-opt')
axes[2].set_xlabel('Nombre de clients (n)')
axes[2].set_ylabel('Amélioration (%)')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.suptitle('Heuristique NNH + 2-opt — Performance par taille d\'instance', 
             y=1.02, fontsize=13)
plt.show()

### 7.3 — Visualisation d'une solution

In [ ]:
def visualiser_solution(routes, instance, titre='Solution heuristique NNH + 2-opt'):
    """
    Affiche les routes sur une carte 2D.
    Chaque véhicule est représenté par une couleur différente.
    Le dépôt est indiqué par une étoile rouge.
    """
    coords = instance['coords']
    colors = plt.cm.tab10.colors

    fig, ax = plt.subplots(figsize=(9, 8))

    for k, route in enumerate(routes):
        if not route:
            continue
        couleur  = colors[k % len(colors)]
        chemin   = [0] + route + [0]  # dépôt → clients → dépôt

        # Tracé de la route
        xs = [coords[c, 0] for c in chemin]
        ys = [coords[c, 1] for c in chemin]
        ax.plot(xs, ys, '-o', color=couleur, linewidth=1.5,
                markersize=5, label=f'Véhicule {k}')

        # Numéros des clients
        for c in route:
            ax.annotate(str(c), coords[c], textcoords='offset points',
                        xytext=(5, 5), fontsize=7, color=couleur)

    # Dépôt
    ax.plot(coords[0, 0], coords[0, 1], '*', color='red',
            markersize=18, zorder=5, label='Dépôt')

    ax.set_title(titre, fontsize=13)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()


# Visualisation sur l'instance 10 clients
instance_20 = generate_instance(n=20, n_vehicles=4, seed=42)
routes_20, cout_20, stats_20 = resoudre_heuristique(instance_20)
visualiser_solution(routes_20, instance_20,
                    titre=f'NNH + 2-opt — 20 clients, coût = {cout_20:.2f}')

### 7.4 — Visualisation networkx (graphe orienté par véhicule)

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import math

def visualiser_networkx(routes, instance, titre='Solution NNH + 2-opt'):
    """
    Affiche chaque tournee dans son propre subplot + un graphe global.
    Chaque sous-graphe montre : depot -> client1 -> ... -> depot.
    """
    coords         = instance['coords']
    palette        = plt.cm.tab10.colors
    routes_actives = [r for r in routes if r]
    K              = len(routes_actives)

    # ── Subplots individuels + 1 graphe global ──────────────────
    ncols = min(3, K + 1)
    nrows = math.ceil((K + 1) / ncols)

    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(6 * ncols, 5 * nrows),
                             squeeze=False)
    fig.suptitle(titre, fontsize=13, fontweight='bold')

    # ── Graphe global (premier subplot) ─────────────────────────
    ax_global = axes[0][0]
    for k, route in enumerate(routes_actives):
        col    = palette[k % len(palette)]
        G_k    = nx.DiGraph()
        pos_k  = {}
        for nd in [0] + route:
            G_k.add_node(nd)
            pos_k[nd] = (coords[nd, 0], coords[nd, 1])
        chemin = [0] + route + [0]
        for a, b in zip(chemin[:-1], chemin[1:]):
            G_k.add_edge(a, b)
        n_edges = G_k.number_of_edges()
        nc = ['#e63946' if nd == 0 else col for nd in G_k.nodes()]
        nx.draw_networkx_nodes(G_k, pos_k, node_color=nc,
                               node_size=[600 if nd==0 else 420 for nd in G_k.nodes()],
                               ax=ax_global)
        nx.draw_networkx_labels(G_k, pos_k, ax=ax_global,
                                font_color='white', font_size=7, font_weight='bold')
        nx.draw_networkx_edges(G_k, pos_k,
                               edge_color=[col] * n_edges,
                               arrows=True, arrowsize=16, width=2.0,
                               ax=ax_global,
                               connectionstyle='arc3,rad=0.10',
                               min_source_margin=18, min_target_margin=18)
    ax_global.set_title('Vue globale — toutes les tournees', fontsize=10)
    ax_global.axis('off')

    # ── Subplots individuels ─────────────────────────────────────
    for k, route in enumerate(routes_actives):
        idx = k + 1   # decale de 1 car le premier est le global
        ax  = axes[idx // ncols][idx % ncols]
        col = palette[k % len(palette)]

        G   = nx.DiGraph()
        pos = {}
        for nd in [0] + route:
            G.add_node(nd)
            pos[nd] = (coords[nd, 0], coords[nd, 1])
        chemin = [0] + route + [0]
        for a, b in zip(chemin[:-1], chemin[1:]):
            G.add_edge(a, b)

        n_edges = G.number_of_edges()
        node_colors = ['#e63946' if nd == 0 else col for nd in G.nodes()]
        node_sizes  = [700 if nd == 0 else 500 for nd in G.nodes()]

        nx.draw_networkx_nodes(G, pos, node_color=node_colors,
                               node_size=node_sizes, ax=ax)
        nx.draw_networkx_labels(G, pos, ax=ax,
                                font_color='white', font_size=8, font_weight='bold')
        nx.draw_networkx_edges(G, pos,
                               edge_color=[col] * n_edges,
                               arrows=True, arrowsize=20, width=2.2,
                               ax=ax,
                               connectionstyle='arc3,rad=0.12',
                               min_source_margin=20, min_target_margin=20)
        ax.set_title(f'Vehicule {k + 1}  ({len(route)} clients)', fontsize=10)
        ax.axis('off')

    # Masquer axes vides
    for k in range(K + 1, nrows * ncols):
        axes[k // ncols][k % ncols].axis('off')

    plt.tight_layout()
    plt.show()

# --- Test sur 3 instances ---
for n_test, seed_test in [(10, 42), (20, 42), (50, 42)]:
    inst    = generate_instance(n=n_test, seed=seed_test)
    r, c, s = resoudre_heuristique(inst)
    visualiser_networkx(r, inst,
        titre=f'NNH + 2-opt — n={n_test}, seed={seed_test} | cout={c:.1f} | valide={s["valide"]}')

### 7.5 — Comparaison NNH seul vs NNH + 2-opt (dataframe)

In [ ]:
import pandas as pd

# Instances de test : 3 tailles x 3 seeds = 9 lignes
configs = [
    (10, 3, 42), (10, 3, 7),  (10, 3, 99),
    (20, 4, 42), (20, 4, 7),  (20, 4, 99),
    (50, 10, 42),(50, 10, 7), (50, 10, 99),
]

lignes = []
for instance_id, (n, k, seed) in enumerate(configs):
    inst = generate_instance(n=n, n_vehicles=k, seed=seed)

    # Etape 1 : NNH seul (sans 2-opt) — on mesure le cout initial
    import time
    t0     = time.time()
    routes_nnh = construire_solution_nnh(inst)
    t_nnh  = round(time.time() - t0, 4)
    cout_nnh = cout_solution(routes_nnh, inst)

    # Etape 2 : NNH + 2-opt complet
    routes_full, cout_2opt, stats = resoudre_heuristique(inst)

    gain = round((cout_nnh - cout_2opt) / cout_nnh * 100, 2) if cout_nnh > 0 else 0.0

    lignes.append({
        'instance_id' : instance_id,
        'n'           : n,
        'seed'        : seed,
        'cout_nnh'    : round(cout_nnh, 2),
        'cout_2opt'   : round(cout_2opt, 2),
        'gain_pct'    : gain,
        'temps_nnh'   : stats['temps_nnh'],
        'temps_2opt'  : stats['temps_2opt'],
        'temps_total' : stats['temps_total'],
        'n_vehicules' : stats['n_vehicules_utilises'],
        'valide'      : stats['valide'],
    })

df = pd.DataFrame(lignes)

# Affichage
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
print(df.to_string(index=False))
print()
print('--- Moyenne gain 2-opt par taille ---')
print(df.groupby('n')['gain_pct'].mean().round(2).to_string())

### 7.6 — Etude statistique : 30 instances x 5 tailles (Phase B)

In [ ]:
# 30 seeds x 5 tailles = 150 runs
TAILLES = [10, 20, 50, 100, 200]
N_RUNS  = 30

lignes_stats = []

for n in TAILLES:
    k = max(3, n // 5)
    for seed in range(N_RUNS):
        inst = generate_instance(n=n, n_vehicles=k, seed=seed)
        _, cout, stats = resoudre_heuristique(inst)
        lignes_stats.append({
            'n'              : n,
            'seed'           : seed,
            'cout'           : round(cout, 2),
            'cout_nnh'       : stats['cout_initial'],
            'cout_2opt'      : stats['cout_final'],
            'gain_pct'       : stats['gain_2opt_pct'],
            'temps_nnh'      : stats['temps_nnh'],
            'temps_2opt'     : stats['temps_2opt'],
            'temps_total'    : stats['temps_total'],
            'n_vehicules'    : stats['n_vehicules_utilises'],
            'n_infaisables'  : stats['n_clients_infaisables'],
            'valide'         : stats['valide'],
        })

df_stats = pd.DataFrame(lignes_stats)

# Agregats par taille
agg = df_stats.groupby('n').agg(
    cout_moyen   = ('cout',       'mean'),
    cout_std     = ('cout',       'std'),
    cout_median  = ('cout',       'median'),
    cout_min     = ('cout',       'min'),
    cout_max     = ('cout',       'max'),
    temps_moyen  = ('temps_total','mean'),
    temps_std    = ('temps_total','std'),
    gain_moyen   = ('gain_pct',   'mean'),
    pct_valide   = ('valide',     'mean'),
).round(3)

agg['pct_valide'] = (agg['pct_valide'] * 100).round(1)

print(agg.to_string())

### 7.7 — Boxplot du coût par taille + courbe temps vs n (log-log)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

TAILLES = [10, 20, 50, 100, 200]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Phase B — Etude statistique NNH + 2-opt (30 instances par taille)',
             fontsize=13, fontweight='bold')

# ── Graphe 1 : Boxplot du cout par taille ───────────────────
ax1 = axes[0]
data_cout = [df_stats[df_stats['n'] == n]['cout'].values for n in TAILLES]
bp = ax1.boxplot(data_cout, tick_labels=TAILLES, patch_artist=True,
                 medianprops={'color': 'black', 'linewidth': 2})
colors = plt.cm.Blues(np.linspace(0.4, 0.85, len(TAILLES)))
for patch, col in zip(bp['boxes'], colors):
    patch.set_facecolor(col)
ax1.set_xlabel('Nombre de clients (n)')
ax1.set_ylabel('Cout total (distance)')
ax1.set_title('Distribution du cout par taille')
ax1.grid(True, alpha=0.3, axis='y')

# ── Graphe 2 : Temps moyen vs n (log-log) ───────────────────
ax2 = axes[1]
agg = df_stats.groupby('n').agg(
    temps_moyen=('temps_total', 'mean'),
    temps_std  =('temps_total', 'std')
).reset_index()

ax2.errorbar(agg['n'], agg['temps_moyen'],
             yerr=agg['temps_std'],
             fmt='o-', color='tomato', linewidth=2,
             markersize=7, capsize=4, label='Temps moyen')
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlabel('n (echelle log)')
ax2.set_ylabel('Temps (s, echelle log)')
ax2.set_title('Temps de calcul vs n (log-log)')
ax2.grid(True, which='both', alpha=0.3)

# Reference O(n²) : caler sur le premier point
t0, n0 = agg['temps_moyen'].iloc[0], agg['n'].iloc[0]
n_ref   = np.array(TAILLES, dtype=float)
ax2.plot(n_ref, t0 * (n_ref / n0) ** 2,
         '--', color='gray', linewidth=1.5, label='Reference O(n²)')
ax2.legend(fontsize=9)

# ── Graphe 3 : Gain 2-opt moyen par taille ──────────────────
ax3 = axes[2]
agg_gain = df_stats.groupby('n')['gain_pct'].mean()
bars = ax3.bar(TAILLES, agg_gain.values,
               color=plt.cm.Greens(np.linspace(0.4, 0.85, len(TAILLES))),
               edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, agg_gain.values):
    ax3.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.02,
             f'{val:.2f}%', ha='center', va='bottom', fontsize=9)
ax3.set_xlabel('Nombre de clients (n)')
ax3.set_ylabel('Gain moyen 2-opt (%)')
ax3.set_title('Gain moyen apporte par le 2-opt')
ax3.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

### 7.8 — Export CSV standardise (`resultats_nnh.csv`)

In [ ]:
import os

# Colonnes standardisees communes a tous les membres de l'equipe
colonnes_export = [
    'methode',       # identifiant de la methode
    'n',             # nombre de clients
    'seed',          # graine aleatoire
    'cout',          # cout final de la solution
    'cout_initial',  # cout NNH seul (avant 2-opt)
    'gain_pct',      # gain apporte par l'amelioration locale
    'temps_total',   # temps de calcul total (secondes)
    'n_vehicules',   # nombre de vehicules utilises
    'n_infaisables', # clients non servis (0 si solution valide)
    'valide',        # True si toutes les contraintes sont respectees
]

# Construction du dataframe d'export a partir de df_stats
df_export = df_stats.copy()
df_export['methode']       = 'NNH+2opt'
df_export['cout_initial']  = df_stats['cout_nnh']
df_export['cout']          = df_stats['cout_2opt']

# Garder uniquement les colonnes standardisees
for col in colonnes_export:
    if col not in df_export.columns:
        df_export[col] = None

df_export = df_export[colonnes_export]

# Export
chemin_csv = os.path.join('..', 'resultats_nnh.csv')
df_export.to_csv(chemin_csv, index=False, encoding='utf-8')

print(f"Export : {chemin_csv}")
print(f"Lignes : {len(df_export)}")
print(f"Colonnes : {list(df_export.columns)}")
print()
print(df_export.head(10).to_string(index=False))

---

## Récapitulatif

| Étape | Méthode | Complexité | Rôle |
|-------|---------|-----------|------|
| 1 | **NNH** | $O(n^2)$ par véhicule | Construire une solution valide rapidement |
| 2 | **2-opt** | $O(m^4)$ par route au pire | Éliminer les croisements, raccourcir les routes |
| — | **Total** | Très rapide en pratique | Baseline fiable pour comparer recuit et DL |

### Apport du 2-opt (attendu d'après la littérature)

- Sans 2-opt : NNH est **~20–25%** au-dessus de l'optimal
- Avec 2-opt : on descend à **~5–10%** au-dessus de l'optimal

### Ce que cette heuristique ne fait pas

- Elle n'échange **pas** de clients entre véhicules (améliorations inter-routes — c'est l'or-opt)
- Elle ne remet **pas** en cause la solution globale (c'est le recuit simulé de Rayene)
- Elle ne **prédit** pas (c'est le Deep Learning de Victor)

C'est pour cela qu'elle sert de **référence minimale** : toute méthode plus complexe doit faire **au moins** aussi bien.